# 24. 다양한 기반 모델 배깅과 보팅 비교 (Bagging & Voting)

동일한 이진 분류 데이터셋에서 KNN, Logistic Regression, Decision Tree 기반 배깅(Bagging) 앙상블과 서로 다른 모델을 결합하는 보팅(VotingClassifier) 성능을 비교합니다.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import BaggingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, n_redundant=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## 문제 1. KNN·LR·DT 기반 배깅 모델 성능 비교

KNN, LogisticRegression, DecisionTree를 각각 기반 모델로 하는 `BaggingClassifier(n_estimators=50, random_state=42)`를 학습하고 테스트 정확도를 비교 출력하세요.

In [ ]:
bag_knn = BaggingClassifier(Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=5))]), n_estimators=50, random_state=42)
bag_lr = BaggingClassifier(Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=500))]), n_estimators=50, random_state=42)
bag_dt = BaggingClassifier(DecisionTreeClassifier(random_state=42), n_estimators=50, random_state=42)
for name, m in [('KNN', bag_knn), ('LR', bag_lr), ('DT', bag_dt)]:
    m.fit(X_train, y_train)
    print(f'Bagging {name}: {accuracy_score(y_test, m.predict(X_test)):.4f}')

**결과 해석**

같은 배깅 기법이라도 분산이 큰 결정 트리나 KNN에 적용할 때 선형 모델(LR)보다 앙상블 분산 감소 효과가 두드러집니다.

## 문제 2. VotingClassifier(soft) 앙상블 학습 및 평가

KNN, LogisticRegression, DecisionTree 세 모델을 `VotingClassifier(voting="soft")`로 결합해 테스트 정확도를 출력하세요.

In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ('knn', Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=5))])),
        ('lr', Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=500))])),
        ('dt', DecisionTreeClassifier(random_state=42)),
    ],
    voting='soft'
)
voting_clf.fit(X_train, y_train)
print(f'Voting (soft) Accuracy: {accuracy_score(y_test, voting_clf.predict(X_test)):.4f}')

**결과 해석**

서로 다른 가설 공간을 가진 모델의 예측 확률을 평균내는 소프트 보팅은 개별 모델의 약점을 상호 보완합니다.